# Taller Práctico 01 — Del dato crudo a la decisión
### Dataset C — Movilidad urbana (sensores IoT de tráfico y clima, Medellín)

**Maestría en Ciencia de Datos y Analítica — Universidad EAFIT**
Fundamentos en Ciencia de Datos | Docente: Jorge Iván Padilla-Buriticá 

**Equipo:**

- Miguel Ángel Cano Salinas
- Daniel Correa Botero
- Daniel Felipe Arango Guarín

---

In [79]:
# Install dependencies
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [80]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [ ]:
clean_df = pd.read_csv('../data/raw/movilidad_sensores_LIMPIO.csv')
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sensor_id         1440 non-null   str    
 1   ubicacion         1440 non-null   str    
 2   tipo_via          1440 non-null   str    
 3   timestamp         1440 non-null   str    
 4   conteo_vehiculos  1440 non-null   int64  
 5   temperatura_c     1440 non-null   float64
 6   condicion_clima   1440 non-null   str    
 7   lat               1440 non-null   float64
 8   lon               1440 non-null   float64
dtypes: float64(3), int64(1), str(5)
memory usage: 168.9 KB


## Inventario de variables - Dataset limpio

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `sensor_id` | Nominal | CSV | Estructurada |
| `ubicacion` | Nominal | CSV | Estructurada |
| `tipo_via` | Nominal | CSV | Estructurada |
| `timestamp` | Fecha | CSV | Estructurada |
| `conteo_vehiculos` | Discreto | CSV | Estructurada |
| `temperatura_c` | Continuo | CSV | Estructurada |
| `condicion_clima` | Nominal | CSV | Estructurada |
| `lat` | Geoespacial | CSV | Estructurada |
| `lon` | Geoespacial | CSV | Estructurada |

In [ ]:
dirty_df = pd.read_csv('../data/raw/movilidad_sensores_CONTAMINADO.csv')
dirty_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1455 entries, 0 to 1454
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sensor_id         1455 non-null   str    
 1   ubicacion         1455 non-null   str    
 2   tipo_via          1455 non-null   str    
 3   timestamp         1455 non-null   str    
 4   conteo_vehiculos  1310 non-null   float64
 5   temperatura_c     1383 non-null   float64
 6   condicion_clima   1368 non-null   str    
 7   lat               1455 non-null   float64
 8   lon               1455 non-null   float64
dtypes: float64(4), str(5)
memory usage: 170.2 KB


## Inventario de variables - Dataset contaminado

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `sensor_id` | Nominal | CSV | Estructurada |
| `ubicacion` | Nominal | CSV | Estructurada |
| `tipo_via` | Nominal | CSV | Estructurada |
| `timestamp` | Fecha | CSV | Estructurada |
| `conteo_vehiculos` | Discreto | CSV | Estructurada |
| `temperatura_c` | Continuo | CSV | Estructurada |
| `condicion_clima` | Nominal | CSV | Estructurada |
| `lat` | Geoespacial | CSV | Estructurada |
| `lon` | Geoespacial | CSV | Estructurada |

In [ ]:
weather_log_df = pd.read_json('../data/raw/clima_api_log.json')
weather_log_flat_df = pd.concat(
    [
        weather_log_df.drop(columns=['location', 'weather']).reset_index(drop=True),
        pd.json_normalize(weather_log_df['location']),
        pd.json_normalize(weather_log_df['weather'])
    ],
    axis=1
)
weather_log_flat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   request_id    120 non-null    str           
 1   timestamp     120 non-null    datetime64[us]
 2   sensor_id     120 non-null    str           
 3   lat           120 non-null    float64       
 4   lon           120 non-null    float64       
 5   temp_c        116 non-null    float64       
 6   condition     120 non-null    str           
 7   humidity_pct  120 non-null    int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(3)
memory usage: 9.6 KB


## Inventario de variables - Dataset JSON

| Variable | Tipo de dato | Formato de origen | Fuente |
|---|---|---|---|
| `request_id` | Nominal | JSON anidado | Semi-estructurada |
| `timestamp` | Fecha | JSON anidado | Semi-estructurada |
| `sensor_id` | Nominal | JSON anidado | Semi-estructurada |
| `lat` | Geoespacial | JSON anidado | Semi-estructurada |
| `lon` | Geoespacial | JSON anidado | Semi-estructurada |
| `temp_c` | Continuo | JSON anidado | Semi-estructurada |
| `condition` | Nominal | JSON anidado | Semi-estructurada |
| `humidity_pct` | Continuo | JSON anidado | Semi-estructurada |

## Reflexión

**¿Qué información se pierde o se distorsiona al forzar una fuente no estructurada/semi-estructurada dentro de una tabla rectangular?**

Se pueden perder la estandarización de los tipos de datos. La relación y jerarquía entre los campos, los cuales pueden brindar información adicional sobre la estructura de los datos. Además, el JSON puede agregar un campo nuevo mañana sin romperse a comparación del CSV. La tabla rectangular exige que todas las filas tengan las mismas columnas, así que cualquier campo y/o opcional nuevo obliga a rellenar con nulos todo lo anterior. 

## Tarea 2 — Diagnóstico GIGO

Funciones usadas en el preprocesamiento para encontrar problemas en los datos.

In [ ]:
# Quick checks to detect data quality issues

print("Null values per column:")
print(dirty_df.isnull().sum())

print("\nUnique values in 'condicion_clima':")
print(dirty_df["condicion_clima"].value_counts(dropna=False))

print("\nDuplicated rows by 'sensor_id' + 'timestamp':")
print(dirty_df.duplicated(subset=["sensor_id", "timestamp"]).sum())

print("\nInvalid timestamps:")
print(pd.to_datetime(dirty_df["timestamp"], errors="coerce").isna().sum())

print("\nNegative values in 'conteo_vehiculos':")
print((dirty_df["conteo_vehiculos"] < 0).sum())

print("\nLongitude summary:")
display(dirty_df["lon"].describe())

print("\nCoordinates outside Medellin range:")
print(((~dirty_df["lat"].between(6.09, 6.39)) | (~dirty_df["lon"].between(-75.7, -75.47))).sum())

print("\nPlaceholder values (>=999) in 'conteo_vehiculos':")
print((dirty_df["conteo_vehiculos"] >= 999).sum())

Null values per column:
sensor_id             0
ubicacion             0
tipo_via              0
timestamp             0
conteo_vehiculos    145
temperatura_c        72
condicion_clima      87
lat                   0
lon                   0
dtype: int64

Unique values in 'condicion_clima':
Soleado     1
Nublado     1
Sol         1
Lluvia      1
Nubes       1
soleado     1
LLUVIA      1
SOLEADO     1
lluvia      1
nublado     1
lluvioso    1
Name: count, dtype: int64

Number of duplicated rows based on 'sensor_id' and 'timestamp':
14

Invalid timestamps:
90

Negative values in 'conteo_vehiculos':
6


count    1455.000000
mean      -68.768157
std        22.612127
min       -75.591600
25%       -75.585105
50%       -75.572430
75%       -75.565360
max         6.287500
Name: lon, dtype: float64


Latitude and Longitude values outside Medellín's expected range 
121

Placeholder values (999) in 'conteo_vehiculos':
4


Hicimos un análisis exploratorio del dataset de **tráfico** y encontramos varios problemas que afectan la calidad de cualquier resultado.

| Problema | Columna(s) | Cómo lo detectamos | Por qué importa |
|---|---|---|---|
| El clima aparece escrito de muchas formas para pocas categorías reales (Sol, soleado, SOLEADO...) | `condicion_clima` | `dirty_df["condicion_clima"].value_counts(dropna=False)` | Si una misma categoría queda fragmentada en etiquetas distintas, el análisis por clima se distorsiona. |
| Hay registros repetidos con el mismo sensor y timestamp | `sensor_id`, `timestamp` | `dirty_df.duplicated(subset=["sensor_id","timestamp"]).sum()` | Un sensor no debería reportar dos lecturas diferentes en el mismo instante; eso introduce inconsistencia. |
| Hay fechas en formatos distintos y varias no se pueden convertir | `timestamp` | `pd.to_datetime(dirty_df["timestamp"], errors="coerce").isna().sum()` | Sin timestamp válido no podemos hacer análisis temporal confiable (horas pico, fines de semana, etc.). |
| Hay conteos vehiculares negativos | `conteo_vehiculos` | `(dirty_df["conteo_vehiculos"] < 0).sum()` | Un conteo de vehículos no puede ser negativo; es ruido del sensor. |
| Hay filas con coordenadas fuera del rango de Medellín | `lat`, `lon` | `((~dirty_df["lat"].between(6.09, 6.39)) | (~dirty_df["lon"].between(-75.7, -75.47))).sum()` | Ubican sensores en lugares incorrectos y afectan cualquier análisis geográfico. |
| Hay valores extremadamente altos (`>= 999`) | `conteo_vehiculos` | `(dirty_df["conteo_vehiculos"] >= 999).sum()` | Estos valores inflan métricas como la media y no representan tráfico real. |
| Hay valores nulos en variables clave | `conteo_vehiculos`, `temperatura_c`, `condicion_clima` | `dirty_df.isnull().sum()` | Si faltan lecturas, podemos perder picos reales o sesgar los resultados. |

# Tarea 3 - Transformación y limpieza con pandas

Corregimos los problemas con este orden: primero eliminamos duplicados exactos y luego duplicados por llave de negocio (`sensor_id` + `timestamp`). Después tratamos fechas inválidas, valores imposibles y coordenadas fuera de Medellín. Al final imputamos nulos cuando el dataset ya estaba estable.

In [ ]:
working_df = dirty_df.copy()  # Keep the raw dirty dataset untouched
working_df

,sensor_id,ubicacion,tipo_via,timestamp,conteo_vehiculos,temperatura_c,condicion_clima,lat,lon
0,SEN03,Calle 10,Local,2025-03-09 16:00:00,31.0,20.7,Soleado,6.20995,-75.57081
1,SEN06,Circular 4ta,Local,2025-03-14 12:00:00,24.0,24.4,Soleado,6.24465,-75.58474
2,SEN04,Autopista Norte,Troncal,2025-03-07 08:00:00,NaN,19.6,Soleado,-75.56538,6.28615
3,SEN01,Av. Regional,Troncal,2025-03-13 04:00:00,16.0,26.3,Soleado,6.23099,-75.58968
4,SEN06,Circular 4ta,Local,2025-03-19 12:00:00,22.0,25.3,Nublado,6.24469,-75.58568
...,...,...,...,...,...,...,...,...,...
1450,SEN01,Av. Regional,Troncal,2025-03-07 10:00:00,13.0,25.2,Soleado,6.23089,-75.59111
1451,SEN04,Autopista Norte,Troncal,2025-03-20 08:00:00,39.0,23.8,Lluvia,-75.56536,6.28674
1452,SEN04,Autopista Norte,Troncal,2025-03-07 04:00:00,19.0,20.8,Nublado,-75.56554,6.28700
1453,SEN06,Circular 4ta,Local,"18 de March de 2025, 20:00",12.0,23.2,Soleado,6.24469,-75.58500


# Etiquetas inconsistentes en condición climática

Encontramos etiquetas inconsistentes en la columna `condicion_clima`: representan lo mismo, pero están escritas de forma diferente.

**Decisión tomada:** Estandarizar con mapeo (no eliminar filas).

Estandarizamos porque el problema no era la categoría en sí, sino la forma de escribirla. Por ejemplo, "Sol", "Soleado" y "soleado" se unifican en una sola categoría.

In [ ]:
weather_label_map = {
    "sol": "Soleado", "soleado": "Soleado",
    "nubes": "Nublado", "nublado": "Nublado",
    "lluvia": "Lluvia", "lluvioso": "Lluvia",
}

working_df["condicion_clima"] = (
    working_df["condicion_clima"]
    .str.strip()
    .str.lower()
    .map(weather_label_map)
)

working_df["condicion_clima"].value_counts(dropna=False)

Soleado    1
Nublado    1
Lluvia     1
Name: count, dtype: int64

# Sensor ID y timestamp repetidos

**Llave de negocio usada:** `sensor_id` + `timestamp`

**¿Por qué?** Un sensor físico solo debería registrar una lectura por instante. Si aparecen varios registros con la misma llave, solo uno puede ser válido. Usar `.duplicated()` sin `subset` solo detecta filas 100% iguales y no captura este caso de negocio.

**Decisión:** conservar el último registro (`keep="last"`).

In [ ]:
working_df = working_df.drop_duplicates(subset=["sensor_id", "timestamp"], keep="last")
working_df

,sensor_id,ubicacion,tipo_via,timestamp,conteo_vehiculos,temperatura_c,condicion_clima,lat,lon
0,SEN03,Calle 10,Local,2025-03-09 16:00:00,31.0,20.7,Soleado,6.20995,-75.57081
1,SEN06,Circular 4ta,Local,2025-03-14 12:00:00,24.0,24.4,Soleado,6.24465,-75.58474
2,SEN04,Autopista Norte,Troncal,2025-03-07 08:00:00,NaN,19.6,Soleado,-75.56538,6.28615
3,SEN01,Av. Regional,Troncal,2025-03-13 04:00:00,16.0,26.3,Soleado,6.23099,-75.58968
4,SEN06,Circular 4ta,Local,2025-03-19 12:00:00,22.0,25.3,Nublado,6.24469,-75.58568
...,...,...,...,...,...,...,...,...,...
1450,SEN01,Av. Regional,Troncal,2025-03-07 10:00:00,13.0,25.2,Soleado,6.23089,-75.59111
1451,SEN04,Autopista Norte,Troncal,2025-03-20 08:00:00,39.0,23.8,Lluvia,-75.56536,6.28674
1452,SEN04,Autopista Norte,Troncal,2025-03-07 04:00:00,19.0,20.8,Nublado,-75.56554,6.28700
1453,SEN06,Circular 4ta,Local,"18 de March de 2025, 20:00",12.0,23.2,Soleado,6.24469,-75.58500


# Fechas en formatos diferentes

**Decisión tomada:** eliminar filas con timestamps inválidos (no imputar).

**¿Por qué?** El `timestamp` no es un dato que podamos estimar de forma confiable. Si no se puede convertir, no sirve para análisis horario (picos, fin de semana, día/noche).

Con `pd.to_datetime(..., errors="coerce")`, los valores inválidos se convierten en `NaT` y luego se eliminan para mantener consistencia temporal.

In [ ]:
invalid_timestamps = pd.to_datetime(working_df["timestamp"], errors="coerce").isna()
print("Fechas que no se pueden convertir:", invalid_timestamps.sum(),
      f"({invalid_timestamps.mean():.1%} del total)")

working_df["timestamp"] = pd.to_datetime(working_df["timestamp"], errors="coerce")
working_df = working_df.dropna(subset=["timestamp"])

working_df

Fechas que no se pueden convertir: 90 (6.2% del total)


,sensor_id,ubicacion,tipo_via,timestamp,conteo_vehiculos,temperatura_c,condicion_clima,lat,lon
0,SEN03,Calle 10,Local,2025-03-09 16:00:00,31.0,20.7,Soleado,6.20995,-75.57081
1,SEN06,Circular 4ta,Local,2025-03-14 12:00:00,24.0,24.4,Soleado,6.24465,-75.58474
2,SEN04,Autopista Norte,Troncal,2025-03-07 08:00:00,NaN,19.6,Soleado,-75.56538,6.28615
3,SEN01,Av. Regional,Troncal,2025-03-13 04:00:00,16.0,26.3,Soleado,6.23099,-75.58968
4,SEN06,Circular 4ta,Local,2025-03-19 12:00:00,22.0,25.3,Nublado,6.24469,-75.58568
...,...,...,...,...,...,...,...,...,...
1449,SEN02,Av. 33,Arteria,2025-03-08 18:00:00,34.0,22.7,Soleado,6.21439,-75.57296
1450,SEN01,Av. Regional,Troncal,2025-03-07 10:00:00,13.0,25.2,Soleado,6.23089,-75.59111
1451,SEN04,Autopista Norte,Troncal,2025-03-20 08:00:00,39.0,23.8,Lluvia,-75.56536,6.28674
1452,SEN04,Autopista Norte,Troncal,2025-03-07 04:00:00,19.0,20.8,Nublado,-75.56554,6.28700


# Conteos vehiculares negativos

**Decisión tomada:** convertir a `NaN`, sin eliminar la fila completa.

**¿Por qué?** Un conteo negativo es un error claro, pero el resto de la fila (ubicación, timestamp, temperatura y clima) puede seguir siendo útil. Al pasar solo ese valor a `NaN`, luego podemos imputarlo.

In [ ]:
negative_counts = working_df["conteo_vehiculos"] < 0
working_df.loc[negative_counts, "conteo_vehiculos"] = np.nan

# Coordenadas fuera de la ciudad

**Decisión tomada:** Descartar filas con coordenadas fuera de rango (no corregir automáticamente).

**¿Cómo definimos el rango geográfico válido?** Se consultaron fuentes web oficiales que establecen que Medellín se encuentra aproximadamente entre:
- Latitud: 6.09° N a 6.39° N
- Longitud: -75.7° W a -75.47° W

**¿Por qué?** No tenemos forma confiable de saber dónde debería estar realmente el sensor.

Intentar "corregir" automáticamente (por ejemplo, intercambiando lat y lon) introduciría suposiciones no verificables. Es más seguro descartar estas 121 filas y trabajar solo con datos geográficamente verificables.

In [ ]:
out_of_range = (~working_df["lat"].between(6.09, 6.39)) | (~working_df["lon"].between(-75.7, -75.47))
working_df = working_df[~out_of_range]

# Valores extremadamente altos

**Decisión tomada:** convertir valores `>= 999` a `NaN`.

**¿Por qué?** En este contexto, esos valores no son realistas y distorsionan métricas como la media.

**¿Por qué no eliminar la fila?** Igual que con los negativos, el resto de variables sigue aportando información útil.

In [ ]:
working_df.loc[working_df["conteo_vehiculos"] >= 999, "conteo_vehiculos"] = np.nan

# Datos nulos

**Estrategia de imputación aplicada:**

**1. `conteo_vehiculos` → mediana con contexto**

- **¿Por qué mediana y no media?** La media se mueve mucho con valores extremos; la mediana representa mejor el valor típico.
- **¿Por qué con contexto (sensor + hora)?** El tráfico cambia por zona y por hora. Primero usamos la mediana del mismo sensor a la misma hora. Si no alcanza, usamos la mediana general del sensor.
- **¿Por qué no eliminar?** Perderíamos muchas filas útiles para el análisis.

**2. `temperatura_c` → mediana por sensor**

- **¿Por qué por sensor y no global?** Cada sensor puede tener condiciones distintas por ubicación.
- **¿Por qué mediana?** Es más robusta frente a lecturas atípicas.

**3. `condicion_clima` → categoría `Desconocido`**

- **¿Por qué `Desconocido` y no moda?** Poner la moda inventa clima que no observamos.
- **¿Por qué no eliminar?** Para varios análisis de movilidad, el clima no es obligatorio.
- **Ventaja práctica:** Después se puede filtrar fácilmente cuando sí se necesite clima válido.

In [ ]:
# 1) conteo_vehiculos -> median by sensor and hour, with sensor fallback
working_df["conteo_vehiculos"] = working_df["conteo_vehiculos"].fillna(
    working_df.groupby([working_df["sensor_id"], working_df["timestamp"].dt.hour])["conteo_vehiculos"].transform("median")
).fillna(working_df.groupby("sensor_id")["conteo_vehiculos"].transform("median"))

# 2) temperatura_c -> median by sensor
working_df["temperatura_c"] = working_df["temperatura_c"].fillna(
    working_df.groupby("sensor_id")["temperatura_c"].transform("median")
)

# 3) condicion_clima -> explicit category for missing values
working_df["condicion_clima"] = working_df["condicion_clima"].fillna("Desconocido")

print("Nulos por columna:\n", working_df[["conteo_vehiculos", "temperatura_c", "condicion_clima"]].isnull().sum())

Nulos por columna:
 conteo_vehiculos    0
temperatura_c       0
condicion_clima     0
dtype: int64


In [ ]:
# Validation checks after cleaning

print("Null values per column:")
print(working_df.isnull().sum())

print("\nUnique values in 'condicion_clima':")
print(working_df["condicion_clima"].value_counts(dropna=False))

print("\nDuplicated rows by 'sensor_id' + 'timestamp':")
print(working_df.duplicated(subset=["sensor_id", "timestamp"]).sum())

print("\nInvalid timestamps:")
print(pd.to_datetime(working_df["timestamp"], errors="coerce").isna().sum())

print("\nNegative values in 'conteo_vehiculos':")
print((working_df["conteo_vehiculos"] < 0).sum())

print("\nCoordinates outside Medellin range:")
print(((~working_df["lat"].between(6.09, 6.39)) | (~working_df["lon"].between(-75.7, -75.47))).sum())

print("\nPlaceholder values (>=999) in 'conteo_vehiculos':")
print((working_df["conteo_vehiculos"] >= 999).sum())

Null values per column:
sensor_id           0
ubicacion           0
tipo_via            0
timestamp           0
conteo_vehiculos    0
temperatura_c       0
condicion_clima     0
lat                 0
lon                 0
dtype: int64

Unique values in 'condicion_clima':
Soleado        1
Nublado        1
Lluvia         1
Desconocido    1
Name: count, dtype: int64

Number of duplicated rows based on 'sensor_id' and 'timestamp':
0

Invalid timestamps:
0

Negative values in 'conteo_vehiculos':
0

Latitude and Longitude values outside Medellín's expected range 
0

Placeholder values (999) in 'conteo_vehiculos':
0


---

# Tarea 4 — Analítica descriptiva cuantitativa, cualitativa y gráfica

Todo lo que sigue se hace sobre `working_df`, que es el dataset que limpiamos en la Tarea 3. El archivo `movilidad_sensores_LIMPIO.csv` solo se usó para validar, no como base del análisis.

Antes de sacar métricas, creamos columnas de apoyo:

- `hora`: hora del día extraída de `timestamp` para analizar horas pico.
- `tipo_dia`: clasifica entre semana y fin de semana.
- `franja`: marcamos como *Pico* las horas 6, 8, 16 y 18 porque fueron las más altas en el promedio por hora; el resto queda como *Valle*.
- `nivel_congestion`: variable objetivo. Definimos **Alta** cuando el conteo es de 40 o más, porque está cerca del percentil 90 (42.5) y es fácil de interpretar.